# Notebook 3: Time-Series Deep Dive

Year-over-year trends beyond what the replication covers. Focus on dynamics, turning points, and crisis impacts.

**Sections:**
1. Asset allocation evolution (all classes, stacked area)
2. Sub-alternative trends (PE, RE, HF, commodities)
3. Performance trends (net returns, with crisis markers)
4. Funding ratio dynamics
5. Hurdle rate compression
6. Membership trends (active vs. retired)
7. Actual vs. target gap
8. Cross-sectional dispersion of omega over time

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'font.size': 11,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'figure.dpi': 100,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
})

COLORS = {
    'blue': '#1f77b4', 'orange': '#d35400', 'purple': '#7f6d9e',
    'light_blue': '#aec7e8', 'red': '#d62728', 'green': '#2ca02c',
    'teal': '#17becf', 'brown': '#8c564b', 'pink': '#e377c2',
}

OUTPUT_DIR = Path('../../output_python/exploration')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Load data
df = pd.read_csv('../../output_python/ppd_plan_level_clean.csv')
perf = pd.read_csv('../../output_python/ppd_performance.csv')

# Derived variables
for prefix in ['Trgt', 'Actl']:
    df[f'alternatives_{prefix.lower()}'] = (
        df[f'PETotal_{prefix}'].fillna(0)
        + df[f'RETotal_{prefix}'].fillna(0)
        + df[f'COMDTotal_{prefix}'].fillna(0)
        + df[f'HFTotal_{prefix}'].fillna(0)
        + df[f'AltMiscTotal_{prefix}'].fillna(0)
        + df[f'OtherTotal_{prefix}'].fillna(0)
    )

df['public_equity_trgt'] = df['EQTotal_Trgt'].fillna(0)
df['fixed_income_trgt'] = df['FITotal_Trgt'].fillna(0) + df['CashTotal_Trgt'].fillna(0)
df['risky_trgt'] = df['public_equity_trgt'] + df['alternatives_trgt']
df['risky_actl'] = df['EQTotal_Actl'].fillna(0) + df['alternatives_actl']
denom = df['alternatives_trgt'] + df['public_equity_trgt']
df['omega'] = np.where(denom > 0, df['alternatives_trgt'] / denom, np.nan)
df['pct_retired'] = np.where(df['TotMembership'] > 0, df['beneficiaries_tot'] / df['TotMembership'], np.nan)

# System-level aggregation
def weighted_avg(group, col, weight_col='MktAssets_net'):
    mask = group[col].notna() & group[weight_col].notna() & (group[weight_col] > 0)
    if mask.sum() == 0:
        return np.nan
    return np.average(group.loc[mask, col], weights=group.loc[mask, weight_col])

agg_cols = [
    'EQTotal_Trgt', 'FITotal_Trgt', 'RETotal_Trgt', 'PETotal_Trgt',
    'HFTotal_Trgt', 'CashTotal_Trgt', 'AltMiscTotal_Trgt', 'COMDTotal_Trgt', 'OtherTotal_Trgt',
    'EQTotal_Actl', 'FITotal_Actl', 'alternatives_trgt', 'alternatives_actl',
    'public_equity_trgt', 'fixed_income_trgt', 'risky_trgt', 'risky_actl',
    'omega', 'ActFundedRatio_GASB', 'InvestmentReturnAssumption_GASB',
    'InvestmentReturn_1yr', 'ret_bgnassets', 'pct_retired',
]

rows = []
for (pub_id, fy), grp in df.groupby(['pub_id', 'fy']):
    row = {'pub_id': pub_id, 'fy': fy}
    for col in agg_cols:
        if col in grp.columns:
            row[col] = weighted_avg(grp, col)
    row['MktAssets_net'] = grp['MktAssets_net'].sum()
    row['TotMembership'] = grp['TotMembership'].sum()
    row['actives_tot'] = grp['actives_tot'].sum()
    row['beneficiaries_tot'] = grp['beneficiaries_tot'].sum()
    rows.append(row)

sys_df = pd.DataFrame(rows)
sys_df['pct_retired'] = np.where(sys_df['TotMembership'] > 0, sys_df['beneficiaries_tot'] / sys_df['TotMembership'], np.nan)

print(f'System-level: {sys_df.shape[0]} obs, {sys_df["pub_id"].nunique()} systems')
print(f'Performance file: {perf.shape[0]} rows')

## Section 1: Asset Allocation Evolution (All Classes)

Stacked area chart showing the full composition shift across 2001–2021.

In [ ]:
# Stacked area: AUM-weighted target allocation by asset class over time
alloc_series_cols = {
    'Public Equity': 'EQTotal_Trgt',
    'Fixed Income': 'FITotal_Trgt',
    'Real Estate': 'RETotal_Trgt',
    'Private Equity': 'PETotal_Trgt',
    'Hedge Funds': 'HFTotal_Trgt',
    'Cash': 'CashTotal_Trgt',
    'Other/Misc': 'AltMiscTotal_Trgt',
}

alloc_ts = []
for yr in range(2001, 2022):
    sub = sys_df[sys_df['fy'] == yr]
    row = {'fy': yr}
    for label, col in alloc_series_cols.items():
        mask = sub[col].notna() & sub['MktAssets_net'].notna() & (sub['MktAssets_net'] > 0)
        if mask.sum() > 0:
            row[label] = np.average(sub.loc[mask, col], weights=sub.loc[mask, 'MktAssets_net']) * 100
        else:
            row[label] = 0
    alloc_ts.append(row)

alloc_df = pd.DataFrame(alloc_ts).set_index('fy')

# Color palette
area_colors = ['#1f77b4', '#2ca02c', '#8c564b', '#9467bd', '#e377c2', '#7f7f7f', '#bcbd22']

fig, ax = plt.subplots(figsize=(12, 6))
ax.stackplot(alloc_df.index, alloc_df.T, labels=alloc_df.columns, colors=area_colors, alpha=0.85)
ax.set_xlabel('Fiscal Year')
ax.set_ylabel('Target Allocation (%)')
ax.set_title('Asset Allocation Evolution (AUM-Weighted Target Shares)')
ax.legend(loc='center left', bbox_to_anchor=(1, 0.5), fontsize=9)
ax.set_xlim(2001, 2021)
ax.set_ylim(0, 100)
ax.grid(axis='y', alpha=0.3)

fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'ts_allocation_stacked.png')
plt.show()

# Print endpoints
print('Allocation shift (AUM-weighted targets):')
for label in alloc_df.columns:
    print(f'  {label:20s}: {alloc_df.loc[2001, label]:5.1f}% -> {alloc_df.loc[2021, label]:5.1f}% '
          f'(change: {alloc_df.loc[2021, label] - alloc_df.loc[2001, label]:+.1f}pp)')

## Section 2: Sub-Alternative Trends

Which sub-class drove the alternatives boom?

In [ ]:
# Sub-alternative time series
sub_alt_cols = {
    'Private Equity': 'PETotal_Trgt',
    'Real Estate': 'RETotal_Trgt',
    'Hedge Funds': 'HFTotal_Trgt',
    'Commodities': 'COMDTotal_Trgt',
    'Other Alternatives': 'AltMiscTotal_Trgt',
}
sub_alt_colors = ['#9467bd', '#8c564b', '#e377c2', '#17becf', '#bcbd22']

sub_alt_ts = []
for yr in range(2001, 2022):
    sub = sys_df[sys_df['fy'] == yr]
    row = {'fy': yr}
    for label, col in sub_alt_cols.items():
        mask = sub[col].notna() & sub['MktAssets_net'].notna() & (sub['MktAssets_net'] > 0)
        if mask.sum() > 0:
            row[label] = np.average(sub.loc[mask, col], weights=sub.loc[mask, 'MktAssets_net']) * 100
        else:
            row[label] = 0
    sub_alt_ts.append(row)

sa_df = pd.DataFrame(sub_alt_ts).set_index('fy')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: line chart
ax = axes[0]
for (label, _), color in zip(sub_alt_cols.items(), sub_alt_colors):
    ax.plot(sa_df.index, sa_df[label], marker='o', markersize=4, lw=2, color=color, label=label)
ax.set_xlabel('Fiscal Year')
ax.set_ylabel('AUM-Weighted Target Share (%)')
ax.set_title('Sub-Alternative Allocation Trends')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
ax.set_xlim(2001, 2021)

# Right: stacked area for sub-alternatives only
ax = axes[1]
ax.stackplot(sa_df.index, sa_df.T, labels=sa_df.columns, colors=sub_alt_colors, alpha=0.85)
ax.set_xlabel('Fiscal Year')
ax.set_ylabel('Target Share (%)')
ax.set_title('Composition of Alternatives')
ax.legend(loc='upper left', fontsize=8)
ax.set_xlim(2001, 2021)
ax.grid(axis='y', alpha=0.3)

fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'ts_sub_alternatives.png')
plt.show()

# Growth decomposition
print('Sub-alternative growth (2001 → 2021):')
for label in sa_df.columns:
    change = sa_df.loc[2021, label] - sa_df.loc[2001, label]
    print(f'  {label:25s}: {sa_df.loc[2001, label]:5.1f}% -> {sa_df.loc[2021, label]:5.1f}% ({change:+.1f}pp)')

## Section 3: Performance Trends

Net returns (`ret_bgnassets`) over time with crisis markers.

In [ ]:
# Performance: ret_bgnassets percentiles by year
perf_ts = []
for yr in range(2001, 2022):
    vals = perf.loc[perf['fy'] == yr, 'ret_bgnassets'].dropna() * 100
    if len(vals) == 0:
        continue
    perf_ts.append({
        'fy': yr,
        'p10': vals.quantile(0.10),
        'p25': vals.quantile(0.25),
        'median': vals.median(),
        'p75': vals.quantile(0.75),
        'p90': vals.quantile(0.90),
        'mean': vals.mean(),
        'n': len(vals),
    })

perf_df = pd.DataFrame(perf_ts)

fig, ax = plt.subplots(figsize=(12, 6))
ax.fill_between(perf_df['fy'], perf_df['p10'], perf_df['p90'], alpha=0.15, color=COLORS['blue'], label='p10–p90')
ax.fill_between(perf_df['fy'], perf_df['p25'], perf_df['p75'], alpha=0.3, color=COLORS['blue'], label='p25–p75')
ax.plot(perf_df['fy'], perf_df['median'], lw=2, color=COLORS['blue'], marker='o', markersize=4, label='Median')
ax.axhline(0, color='black', ls='-', lw=0.5)

# Crisis markers
crisis_events = {2002: 'Dot-com\nbust', 2008: 'GFC\ncrisis', 2009: 'GFC\nrecovery', 2020: 'COVID'}
for yr, label in crisis_events.items():
    if yr in perf_df['fy'].values:
        med = perf_df.loc[perf_df['fy'] == yr, 'median'].values[0]
        ax.annotate(label, (yr, med), textcoords="offset points",
                    xytext=(0, -25 if med > 0 else 25), fontsize=8, ha='center',
                    arrowprops=dict(arrowstyle='->', color='red', lw=1), color='red')

ax.set_xlabel('Fiscal Year')
ax.set_ylabel('Net Return — ret_bgnassets (%)')
ax.set_title('Pension Fund Performance Over Time')
ax.legend(loc='upper left')
ax.grid(alpha=0.3)
ax.set_xlim(2001, 2021)

fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'ts_performance.png')
plt.show()

# Key stats
print('Performance highlights:')
for yr in [2001, 2008, 2009, 2020, 2021]:
    row = perf_df[perf_df['fy'] == yr]
    if len(row) > 0:
        r = row.iloc[0]
        print(f'  FY{yr}: median={r["median"]:.1f}%, IQR=[{r["p25"]:.1f}%, {r["p75"]:.1f}%], N={int(r["n"])}')

## Section 4: Funding Ratio Dynamics

In [ ]:
# Funding ratio over time with IQR band
fund_ts = []
for yr in range(2001, 2022):
    vals = sys_df.loc[sys_df['fy'] == yr, 'ActFundedRatio_GASB'].dropna() * 100
    if len(vals) == 0:
        continue
    fund_ts.append({
        'fy': yr, 'p25': vals.quantile(0.25), 'median': vals.median(),
        'p75': vals.quantile(0.75), 'p10': vals.quantile(0.10), 'p90': vals.quantile(0.90),
    })

fund_df = pd.DataFrame(fund_ts)

fig, ax = plt.subplots(figsize=(12, 6))
ax.fill_between(fund_df['fy'], fund_df['p10'], fund_df['p90'], alpha=0.15, color=COLORS['green'])
ax.fill_between(fund_df['fy'], fund_df['p25'], fund_df['p75'], alpha=0.3, color=COLORS['green'], label='IQR')
ax.plot(fund_df['fy'], fund_df['median'], lw=2, color=COLORS['green'], marker='o', markersize=4, label='Median')
ax.axhline(100, color='red', ls='--', lw=1, alpha=0.5, label='Fully funded')
ax.axhline(80, color='orange', ls='--', lw=1, alpha=0.5, label='80% threshold')

# Key events
events = {
    2008: 'GFC', 2014: 'GASB 67/68\nadopted', 2020: 'COVID',
}
for yr, label in events.items():
    ax.axvline(yr, color='gray', ls=':', alpha=0.4)
    ax.text(yr, ax.get_ylim()[1] if ax.get_ylim()[1] > 0 else 120, label,
            ha='center', va='bottom', fontsize=8, color='gray')

ax.set_xlabel('Fiscal Year')
ax.set_ylabel('GASB Funded Ratio (%)')
ax.set_title('Funding Ratio Dynamics (2001–2021)')
ax.legend(loc='lower left')
ax.grid(alpha=0.3)
ax.set_xlim(2001, 2021)

fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'ts_funding_ratio.png')
plt.show()

## Section 5: Hurdle Rate Compression

Assumed return distribution over time.

In [ ]:
# Hurdle rate (assumed return) distribution over time
hurdle_ts = []
for yr in range(2001, 2022):
    vals = sys_df.loc[sys_df['fy'] == yr, 'InvestmentReturnAssumption_GASB'].dropna()
    if len(vals) == 0:
        continue
    hurdle_ts.append({
        'fy': yr, 'p10': vals.quantile(0.10), 'p25': vals.quantile(0.25),
        'median': vals.median(), 'p75': vals.quantile(0.75),
        'p90': vals.quantile(0.90), 'mean': vals.mean(),
    })

hurdle_df = pd.DataFrame(hurdle_ts)

fig, ax = plt.subplots(figsize=(12, 5))
ax.fill_between(hurdle_df['fy'], hurdle_df['p10'], hurdle_df['p90'], alpha=0.15, color=COLORS['purple'])
ax.fill_between(hurdle_df['fy'], hurdle_df['p25'], hurdle_df['p75'], alpha=0.3, color=COLORS['purple'], label='IQR')
ax.plot(hurdle_df['fy'], hurdle_df['median'], lw=2, color=COLORS['purple'], marker='o', markersize=4, label='Median')
ax.plot(hurdle_df['fy'], hurdle_df['mean'], lw=1.5, color=COLORS['orange'], ls='--', label='Mean')

ax.set_xlabel('Fiscal Year')
ax.set_ylabel('Assumed Return (%)')
ax.set_title('Hurdle Rate Compression (2001–2021)')
ax.legend()
ax.grid(alpha=0.3)
ax.set_xlim(2001, 2021)

fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'ts_hurdle_rate.png')
plt.show()

print(f'Hurdle rate decline: {hurdle_df.iloc[0]["median"]:.2f}% (2001) -> {hurdle_df.iloc[-1]["median"]:.2f}% (2021)')
print(f'Cross-sectional spread (p10-p90): {hurdle_df.iloc[-1]["p90"]-hurdle_df.iloc[-1]["p10"]:.2f}pp in 2021 vs {hurdle_df.iloc[0]["p90"]-hurdle_df.iloc[0]["p10"]:.2f}pp in 2001')

## Section 6: Membership Trends

Total membership, active vs. retired. The maturation of public pensions.

In [ ]:
# Membership trends
member_ts = []
for yr in range(2001, 2022):
    sub = sys_df[sys_df['fy'] == yr]
    member_ts.append({
        'fy': yr,
        'total': sub['TotMembership'].sum() / 1e6,
        'actives': sub['actives_tot'].sum() / 1e6,
        'retirees': sub['beneficiaries_tot'].sum() / 1e6,
        'pct_retired_median': sub['pct_retired'].dropna().median() * 100,
        'pct_retired_mean': sub['pct_retired'].dropna().mean() * 100,
    })

mem_df = pd.DataFrame(member_ts)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Membership counts
ax = axes[0]
ax.plot(mem_df['fy'], mem_df['total'], lw=2, color=COLORS['blue'], marker='o', markersize=4, label='Total')
ax.plot(mem_df['fy'], mem_df['actives'], lw=2, color=COLORS['green'], marker='s', markersize=4, label='Active')
ax.plot(mem_df['fy'], mem_df['retirees'], lw=2, color=COLORS['red'], marker='^', markersize=4, label='Retirees')
ax.set_xlabel('Fiscal Year')
ax.set_ylabel('Members (millions)')
ax.set_title('Pension Membership Over Time')
ax.legend()
ax.grid(alpha=0.3)
ax.set_xlim(2001, 2021)

# Right: Percent retired
ax = axes[1]
ax.plot(mem_df['fy'], mem_df['pct_retired_median'], lw=2, color=COLORS['red'], marker='o', markersize=4, label='Median')
ax.plot(mem_df['fy'], mem_df['pct_retired_mean'], lw=1.5, color=COLORS['orange'], ls='--', label='Mean')
ax.set_xlabel('Fiscal Year')
ax.set_ylabel('Percent Retired (%)')
ax.set_title('System Maturation: Share of Retirees')
ax.legend()
ax.grid(alpha=0.3)
ax.set_xlim(2001, 2021)

fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'ts_membership.png')
plt.show()

print(f'Pct retired (median): {mem_df.iloc[0]["pct_retired_median"]:.0f}% (2001) -> {mem_df.iloc[-1]["pct_retired_median"]:.0f}% (2021)')

## Section 7: Actual vs. Target Gap

When do pensions deviate most from their targets?

In [ ]:
# Actual – Target risky share gap over time
sys_df['risky_gap'] = sys_df['risky_actl'] - sys_df['risky_trgt']

gap_ts = []
for yr in range(2001, 2022):
    vals = sys_df.loc[sys_df['fy'] == yr, 'risky_gap'].dropna() * 100
    if len(vals) == 0:
        continue
    gap_ts.append({
        'fy': yr, 'p25': vals.quantile(0.25), 'median': vals.median(),
        'p75': vals.quantile(0.75), 'mean': vals.mean(),
    })

gap_df = pd.DataFrame(gap_ts)

fig, ax = plt.subplots(figsize=(12, 5))
ax.fill_between(gap_df['fy'], gap_df['p25'], gap_df['p75'], alpha=0.3, color=COLORS['blue'], label='IQR')
ax.plot(gap_df['fy'], gap_df['median'], lw=2, color=COLORS['blue'], marker='o', markersize=4, label='Median')
ax.axhline(0, color='black', ls='-', lw=0.5)

# Annotate years with large gaps
for yr in [2008, 2009, 2020]:
    row = gap_df[gap_df['fy'] == yr]
    if len(row) > 0:
        med = row.iloc[0]['median']
        ax.annotate(f'FY{yr}\n{med:+.1f}pp', (yr, med), textcoords="offset points",
                    xytext=(15, 0), fontsize=8, color=COLORS['red'],
                    arrowprops=dict(arrowstyle='->', color=COLORS['red'], lw=1))

ax.set_xlabel('Fiscal Year')
ax.set_ylabel('Actual - Target Risky Share (pp)')
ax.set_title('Risky Share: Actual vs. Target Gap Over Time')
ax.legend()
ax.grid(alpha=0.3)
ax.set_xlim(2001, 2021)

fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'ts_actual_target_gap.png')
plt.show()

## Section 8: Cross-Sectional Dispersion

Is the pension universe converging or diverging in allocation strategy?

In [ ]:
# Cross-sectional standard deviation of key variables over time
disp_vars = {
    'Omega (alt-to-risky)': 'omega',
    'Risky share': 'risky_trgt',
    'Alternatives share': 'alternatives_trgt',
    'Funding ratio': 'ActFundedRatio_GASB',
}

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
axes = axes.flatten()

for idx, (label, col) in enumerate(disp_vars.items()):
    ax = axes[idx]
    disp_ts = []
    for yr in range(2001, 2022):
        vals = sys_df.loc[sys_df['fy'] == yr, col].dropna()
        if len(vals) > 5:
            disp_ts.append({
                'fy': yr, 'std': vals.std() * 100, 'iqr': (vals.quantile(0.75) - vals.quantile(0.25)) * 100,
                'cv': vals.std() / vals.mean() if vals.mean() != 0 else np.nan,
            })
    disp_df = pd.DataFrame(disp_ts)

    ax.plot(disp_df['fy'], disp_df['std'], lw=2, color=COLORS['blue'], marker='o', markersize=4, label='Std Dev')
    ax.plot(disp_df['fy'], disp_df['iqr'], lw=2, color=COLORS['orange'], marker='s', markersize=4, label='IQR')
    ax.set_xlabel('Fiscal Year')
    ax.set_ylabel('Dispersion (pp)')
    ax.set_title(f'{label}')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
    ax.set_xlim(2001, 2021)

fig.suptitle('Cross-Sectional Dispersion Over Time', fontsize=14)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'ts_dispersion.png')
plt.show()